# 第07章：Fused MoE Triton GEMM — 从零到 vLLM 级实现

## 本章内容

深入 vLLM 中 **最复杂** 的 Triton 矩阵乘法算子 — `fused_moe_kernel`：

1. MoE 为什么需要自定义 GEMM？(cuBLAS 的局限)
2. Token 路由与排序机制
3. 从零实现 Fused MoE GEMM (5 个版本递进)
4. 量化支持 (FP8 block-wise)
5. 对比 vLLM 的实际实现

## 前置知识
- Notebook 06 的矩阵乘法全景
- Notebook 03 的融合算子与 Autotune
- Ch.12-18 的 Triton GEMM 优化技巧（来自 `03_matmul_optimization` 项目）

## 7.1 MoE 的计算模式 — 为什么 cuBLAS 不够用？

### Mixture of Experts (MoE) 的计算流程

```
输入: hidden_states [num_tokens, hidden_dim]

Step 1: Router (门控网络)
  router_logits = hidden @ W_gate           # [num_tokens, num_experts]
  topk_weights, topk_ids = topk(softmax(router_logits), k=top_k)
  # topk_ids: 每个 token 选择的 expert 索引
  # topk_weights: 对应的路由权重

Step 2: Expert GEMM (关键!)
  for each token i:
    for each selected expert j in topk_ids[i]:
      out[i] += topk_weights[i,j] * (hidden[i] @ Expert_W[j])
  # ↑ 这不是标准 GEMM! 每行的权重矩阵不同!

Step 3: Reduce
  output = sum over top_k experts
```

### cuBLAS 无法处理的原因

```
标准 GEMM:     C = A × B
               所有行共用同一个 B 矩阵 ✓

MoE GEMM:     C[i] = A[i] × B[expert_id[i]]
               每行可能用不同的 B 矩阵 ✗
               → cuBLAS 不支持 "per-row expert selection"
               → 必须自定义 kernel

可选方案:
  ① 循环调用 cuBLAS (每个 expert 一次) → 太多 kernel launch
  ② Batched GEMM (每个 expert 一个 batch) → 需要 scatter/gather, 内存搬运
  ③ 自定义 Triton kernel → 在一个 kernel 中完成路由 + GEMM ← vLLM 的选择
```

In [ ]:
import torch
import triton
import triton.language as tl
import time

print("本 notebook 将从零实现 5 个版本的 MoE GEMM:")
print("  v1: Naive — 循环调用标准 GEMM (基线)")
print("  v2: Grouped Batch — 按 expert 分组 batch GEMM")
print("  v3: Triton Basic — 最简 Triton MoE kernel")
print("  v4: Triton + Swizzle + FP32累加 — 优化版")
print("  v5: vLLM 级 — sorted tokens + routing weight + 量化")

torch.manual_seed(42)
device = 'cuda'

## 7.2 准备数据 — 模拟 MoE 路由

先创建一个简单的 MoE 场景: 8 个 experts, top_k=2

In [ ]:
# MoE 参数设置
num_tokens = 256          # batch 中的 token 数
hidden_dim = 4096         # 隐藏维度 (输入特征)
intermediate_dim = 2048   # 中间维度 (输出特征)
num_experts = 8           # expert 数量
top_k = 2                 # 每个 token 选择的 expert 数

# 输入数据
hidden_states = torch.randn(num_tokens, hidden_dim, device=device, dtype=torch.float16)

# Expert 权重: [num_experts, intermediate_dim, hidden_dim]
# 注意 vLLM 中 B 的 shape 是 [E, N, K], 即 [experts, out_dim, in_dim]
expert_weights = torch.randn(num_experts, intermediate_dim, hidden_dim,
                             device=device, dtype=torch.float16)

# 模拟路由: 每个 token 选 top_k 个 experts
router_logits = torch.randn(num_tokens, num_experts, device=device, dtype=torch.float32)
topk_weights, topk_ids = torch.topk(torch.softmax(router_logits, dim=-1), top_k, dim=-1)
topk_weights = topk_weights.to(torch.float16)

print(f"输入 hidden_states: {hidden_states.shape}")
print(f"Expert 权重: {expert_weights.shape}")
print(f"topk_ids: {topk_ids.shape}, 值范围: [{topk_ids.min()}, {topk_ids.max()}]")
print(f"topk_weights: {topk_weights.shape}")
print()
print(f"路由示例 (前 5 个 token):")
for i in range(5):
    experts = topk_ids[i].tolist()
    weights = topk_weights[i].tolist()
    print(f"  Token {i}: expert {experts}, weights [{weights[0]:.3f}, {weights[1]:.3f}]")

## 7.3 Version 1: Naive — 循环调用 GEMM (基线)

最简单但最慢的实现: 对每个 token 的每个 expert 分别调用 cuBLAS。

In [ ]:
def moe_naive(hidden_states, expert_weights, topk_ids, topk_weights):
    """Naive MoE: per-token, per-expert GEMM"""
    num_tokens = hidden_states.shape[0]
    out_dim = expert_weights.shape[1]
    output = torch.zeros(num_tokens, out_dim, device=hidden_states.device,
                         dtype=hidden_states.dtype)

    for token_idx in range(num_tokens):
        for k in range(topk_ids.shape[1]):
            expert_id = topk_ids[token_idx, k].item()
            weight = topk_weights[token_idx, k]
            # 单个 token × 单个 expert 的 GEMM
            # [1, hidden_dim] × [hidden_dim, out_dim] → [1, out_dim]
            token_out = hidden_states[token_idx:token_idx+1] @ expert_weights[expert_id].T
            output[token_idx] += weight * token_out.squeeze(0)

    return output

# 验证正确性 (小数据)
ref_output = moe_naive(hidden_states[:8], expert_weights, topk_ids[:8], topk_weights[:8])
print(f"Naive output shape: {ref_output.shape}")
print(f"Naive output[0, :5]: {ref_output[0, :5]}")

# Benchmark (只测 8 个 tokens, 否则太慢)
torch.cuda.synchronize()
t0 = time.time()
for _ in range(10):
    moe_naive(hidden_states[:8], expert_weights, topk_ids[:8], topk_weights[:8])
torch.cuda.synchronize()
naive_time = (time.time() - t0) / 10
print(f"\nNaive 8 tokens: {naive_time*1000:.2f} ms")
print(f"估算 {num_tokens} tokens: {naive_time*1000 * num_tokens/8:.1f} ms")
print("\n问题: 每个 token 的每个 expert 产生一次 kernel launch")
print(f"  总 kernel 数 = {num_tokens} × {top_k} = {num_tokens * top_k}")

## 7.4 Version 2: Grouped Batch — 按 Expert 分组

关键思想: 把所有被路由到同一个 expert 的 token **聚合**到一起，做一次大 GEMM。

In [ ]:
def moe_grouped(hidden_states, expert_weights, topk_ids, topk_weights):
    """Grouped MoE: 按 expert 分组, 每个 expert 一次 GEMM"""
    num_tokens, hidden_dim = hidden_states.shape
    out_dim = expert_weights.shape[1]
    num_experts = expert_weights.shape[0]
    top_k = topk_ids.shape[1]

    output = torch.zeros(num_tokens, out_dim, device=hidden_states.device,
                         dtype=hidden_states.dtype)

    for expert_id in range(num_experts):
        # 找到所有选择了这个 expert 的 (token, k) 对
        mask = (topk_ids == expert_id)  # [num_tokens, top_k]
        token_indices, k_indices = torch.where(mask)

        if len(token_indices) == 0:
            continue

        # 聚合 tokens
        expert_input = hidden_states[token_indices]  # [num_selected, hidden_dim]
        expert_weight = expert_weights[expert_id]     # [out_dim, hidden_dim]

        # 一次大 GEMM! [num_selected, hidden_dim] × [hidden_dim, out_dim]
        expert_output = expert_input @ expert_weight.T

        # 加权写回
        weights = topk_weights[token_indices, k_indices]  # [num_selected]
        output[token_indices] += weights.unsqueeze(1) * expert_output

    return output

# 验证
grouped_output = moe_grouped(hidden_states[:8], expert_weights, topk_ids[:8], topk_weights[:8])
diff = (grouped_output - ref_output).abs().max()
print(f"Grouped vs Naive max diff: {diff:.6f}")

# Benchmark 全量
torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    moe_grouped(hidden_states, expert_weights, topk_ids, topk_weights)
torch.cuda.synchronize()
grouped_time = (time.time() - t0) / 100
print(f"\nGrouped {num_tokens} tokens: {grouped_time*1000:.2f} ms")
print(f"只需 {num_experts} 次 GEMM (每个 expert 一次)")
print("\n仍有问题:")
print("  1. 需要 scatter/gather (torch.where + 索引)")
print("  2. 每个 expert 的 batch 大小不均匀")
print("  3. 还是多次 kernel launch")

## 7.5 Version 3: Triton 基础版 — 单 Kernel MoE GEMM

关键创新: **在一个 Triton kernel 中完成 token 路由 + GEMM + 加权**。

### Token 排序预处理

vLLM 在 kernel 外部将 tokens 按 expert 排序，得到 `sorted_token_ids` 和 `expert_ids`：

In [ ]:
def sort_tokens_by_expert(topk_ids, num_experts, block_size_m=32):
    """
    将 tokens 按 expert 排序, 生成 sorted_token_ids 和 expert_ids.
    这是 vLLM moe_align_block_size 的简化版本.
    """
    num_tokens, top_k = topk_ids.shape

    # 展开: 每个 token 出现 top_k 次
    # token_id = token_idx * top_k + k
    flat_topk_ids = topk_ids.view(-1)   # [num_tokens * top_k]
    flat_token_ids = torch.arange(num_tokens * top_k, device=topk_ids.device)

    # 按 expert_id 排序
    sorted_indices = torch.argsort(flat_topk_ids, stable=True)
    sorted_token_ids = flat_token_ids[sorted_indices]
    sorted_expert_ids = flat_topk_ids[sorted_indices]

    # 按 BLOCK_SIZE_M 对齐, 为每个 block 分配 expert
    total = sorted_token_ids.shape[0]
    padded_total = ((total + block_size_m - 1) // block_size_m) * block_size_m

    # Padding
    padded_sorted_ids = torch.full((padded_total,), total,
                                    device=topk_ids.device, dtype=torch.long)
    padded_sorted_ids[:total] = sorted_token_ids

    # Expert ids per block
    num_blocks = padded_total // block_size_m
    expert_ids_per_block = torch.full((num_blocks,), -1,
                                      device=topk_ids.device, dtype=torch.long)
    for i in range(num_blocks):
        start = i * block_size_m
        if start < total:
            expert_ids_per_block[i] = sorted_expert_ids[min(start, total-1)]

    return padded_sorted_ids, expert_ids_per_block, torch.tensor([padded_total], device=topk_ids.device)

# 演示
sorted_ids, expert_ids, num_padded = sort_tokens_by_expert(topk_ids, num_experts, block_size_m=32)
print(f"sorted_token_ids shape: {sorted_ids.shape}")
print(f"expert_ids (per block) shape: {expert_ids.shape}")
print(f"num_tokens_post_padded: {num_padded.item()}")
print()
print("排序后的 token 分布 (前 64 个):")
for i in range(min(2, expert_ids.shape[0])):
    block_tokens = sorted_ids[i*32:(i+1)*32]
    valid = block_tokens[block_tokens < num_tokens * top_k]
    print(f"  Block {i} (Expert {expert_ids[i].item()}): {len(valid)} valid tokens")

### Triton 基础 MoE Kernel

这是最简版本，只有核心的路由 + GEMM 逻辑：

In [ ]:
@triton.jit
def moe_kernel_v3(
    # 指针
    a_ptr,             # [num_tokens, K]
    b_ptr,             # [num_experts, N, K]
    c_ptr,             # [num_tokens * top_k, N]
    sorted_token_ids_ptr,
    expert_ids_ptr,
    num_tokens_post_padded_ptr,
    # 维度
    N, K, EM,
    num_valid_tokens,
    # Strides
    stride_am, stride_ak,
    stride_be, stride_bn, stride_bk,
    stride_cm, stride_cn,
    top_k: tl.constexpr,
    # Meta
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    """
    Fused MoE GEMM (基础版):
    对于每个 M-block:
      1. 从 sorted_token_ids 查找真实 token 位置
      2. 从 expert_ids 查找对应的 expert
      3. 用该 expert 的权重做 GEMM
    """
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(EM, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)

    # ===== Swizzle 调度 (与 Ch.13 相同) =====
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    # ===== 边界检查 =====
    num_tokens_post_padded = tl.load(num_tokens_post_padded_ptr)
    if pid_m * BLOCK_M >= num_tokens_post_padded:
        return

    # ===== 关键: 从 sorted_token_ids 获取真实 token 索引 =====
    offs_m = tl.arange(0, BLOCK_M)
    offs_token_id = pid_m * BLOCK_M + offs_m
    # sorted_token_ids 存的是 "token_idx * top_k + k_idx"
    offs_token = tl.load(sorted_token_ids_ptr + offs_token_id)
    token_mask = offs_token < num_valid_tokens

    # ===== 关键: 获取该 block 的 expert id =====
    off_expert = tl.load(expert_ids_ptr + pid_m)

    # ===== 构建 A 和 B 的指针 =====
    offs_k = tl.arange(0, BLOCK_K)
    offs_n = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N

    # A: 通过 offs_token // top_k 找到原始 token 位置
    a_ptrs = a_ptr + (offs_token[:, None] // top_k * stride_am +
                       offs_k[None, :] * stride_ak)

    # B: 通过 off_expert 选择 expert 权重
    b_ptrs = (b_ptr + off_expert * stride_be +
              offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    # ===== GEMM 主循环 (与标准 GEMM 完全相同!) =====
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs,
                     mask=token_mask[:, None] & (offs_k[None, :] < K - k * BLOCK_K),
                     other=0.0)
        b = tl.load(b_ptrs,
                     mask=offs_k[:, None] < K - k * BLOCK_K,
                     other=0.0)
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # ===== 存储 =====
    acc = acc.to(tl.float16)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + offs_token[:, None] * stride_cm + offs_cn[None, :] * stride_cn
    c_mask = token_mask[:, None] & (offs_cn[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)

print("V3 kernel 定义完成")
print("核心思想: 标准 GEMM 循环 + sorted_token_ids 间接寻址 + expert_ids 权重选择")

In [ ]:
def moe_triton_v3(hidden_states, expert_weights, topk_ids, topk_weights):
    """使用 Triton kernel 的 MoE GEMM"""
    num_tokens, hidden_dim = hidden_states.shape
    num_experts, out_dim, _ = expert_weights.shape
    top_k = topk_ids.shape[1]

    BLOCK_M = 32

    # 预处理: 排序 tokens
    sorted_ids, expert_ids_block, num_padded = sort_tokens_by_expert(
        topk_ids, num_experts, BLOCK_M)

    # 输出: [num_tokens * top_k, out_dim]
    output = torch.zeros(num_tokens * top_k, out_dim,
                         device=hidden_states.device, dtype=torch.float16)

    EM = sorted_ids.shape[0]
    num_valid = num_tokens * top_k

    grid = lambda META: (
        triton.cdiv(EM, META['BLOCK_M']) * triton.cdiv(out_dim, META['BLOCK_N']),
    )

    moe_kernel_v3[grid](
        hidden_states, expert_weights, output,
        sorted_ids, expert_ids_block, num_padded,
        out_dim, hidden_dim, EM,
        num_valid,
        hidden_states.stride(0), hidden_states.stride(1),
        expert_weights.stride(0), expert_weights.stride(1), expert_weights.stride(2),
        output.stride(0), output.stride(1),
        top_k=top_k,
        BLOCK_M=BLOCK_M, BLOCK_N=64, BLOCK_K=32,
        GROUP_SIZE_M=8,
    )

    # 后处理: 加权 reduce
    output = output.view(num_tokens, top_k, out_dim)
    weights = topk_weights.unsqueeze(-1)  # [num_tokens, top_k, 1]
    result = (output * weights).sum(dim=1)  # [num_tokens, out_dim]

    return result

# 验证
v3_output = moe_triton_v3(hidden_states[:8], expert_weights, topk_ids[:8], topk_weights[:8])
diff = (v3_output - ref_output).abs().max()
print(f"V3 Triton vs Naive max diff: {diff:.6f}")
assert diff < 0.1, f"差异过大: {diff}"
print("✓ V3 验证通过")

# Benchmark
torch.cuda.synchronize()
for _ in range(10):
    moe_triton_v3(hidden_states, expert_weights, topk_ids, topk_weights)
torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    moe_triton_v3(hidden_states, expert_weights, topk_ids, topk_weights)
torch.cuda.synchronize()
v3_time = (time.time() - t0) / 100
print(f"\nV3 Triton {num_tokens} tokens: {v3_time*1000:.3f} ms")

## 7.6 Version 4: + Router Weight 融合

vLLM 的 `fused_moe_kernel` 把 **router weight 乘法也融合进 kernel**，
避免了额外的 element-wise kernel launch。

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64, 'BLOCK_K': 32, 'GROUP_SIZE_M': 8},
                      num_stages=3, num_warps=4),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64, 'BLOCK_K': 32, 'GROUP_SIZE_M': 8},
                      num_stages=3, num_warps=4),
        triton.Config({'BLOCK_M': 32, 'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_SIZE_M': 8},
                      num_stages=3, num_warps=4),
        triton.Config({'BLOCK_M': 64, 'BLOCK_N': 128, 'BLOCK_K': 32, 'GROUP_SIZE_M': 8},
                      num_stages=4, num_warps=4),
    ],
    key=['N', 'K'],
)
@triton.jit
def moe_kernel_v4(
    a_ptr, b_ptr, c_ptr,
    topk_weights_ptr,           # 新增: router 权重
    sorted_token_ids_ptr,
    expert_ids_ptr,
    num_tokens_post_padded_ptr,
    N, K, EM,
    num_valid_tokens,
    stride_am, stride_ak,
    stride_be, stride_bn, stride_bk,
    stride_cm, stride_cn,
    top_k: tl.constexpr,
    MUL_ROUTED_WEIGHT: tl.constexpr,  # 是否融合路由权重
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(EM, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)

    # Swizzle
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    num_tokens_post_padded = tl.load(num_tokens_post_padded_ptr)
    if pid_m * BLOCK_M >= num_tokens_post_padded:
        return

    offs_m = tl.arange(0, BLOCK_M)
    offs_token_id = pid_m * BLOCK_M + offs_m
    offs_token = tl.load(sorted_token_ids_ptr + offs_token_id)
    token_mask = offs_token < num_valid_tokens

    off_expert = tl.load(expert_ids_ptr + pid_m)
    if off_expert == -1:
        return

    offs_k = tl.arange(0, BLOCK_K)
    offs_n = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N

    a_ptrs = a_ptr + (offs_token[:, None] // top_k * stride_am +
                       offs_k[None, :] * stride_ak)
    b_ptrs = (b_ptr + off_expert * stride_be +
              offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    # ===== GEMM + FP32 累加 =====
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs,
                     mask=token_mask[:, None] & (offs_k[None, :] < K - k * BLOCK_K),
                     other=0.0)
        b = tl.load(b_ptrs,
                     mask=offs_k[:, None] < K - k * BLOCK_K,
                     other=0.0)
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # ===== 新增: 在 kernel 内部乘 router weight =====
    if MUL_ROUTED_WEIGHT:
        moe_weight = tl.load(topk_weights_ptr + offs_token,
                             mask=token_mask, other=0.0)
        acc *= moe_weight[:, None]  # broadcast: [BLOCK_M, 1] * [BLOCK_M, BLOCK_N]

    acc = acc.to(tl.float16)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + offs_token[:, None] * stride_cm + offs_cn[None, :] * stride_cn
    c_mask = token_mask[:, None] & (offs_cn[None, :] < N)
    tl.store(c_ptrs, acc, mask=c_mask)

print("V4: + Router weight 融合")
print("优势: 省掉了 GEMM 后的 element-wise 乘法 kernel")
print("在 FP32 精度下乘 weight, 然后转 FP16 存储 → 精度更好")

In [ ]:
def moe_triton_v4(hidden_states, expert_weights, topk_ids, topk_weights):
    """V4: Triton MoE with fused router weight"""
    num_tokens, hidden_dim = hidden_states.shape
    num_experts, out_dim, _ = expert_weights.shape
    top_k = topk_ids.shape[1]

    BLOCK_M = 32  # autotune 会覆盖
    sorted_ids, expert_ids_block, num_padded = sort_tokens_by_expert(
        topk_ids, num_experts, BLOCK_M)

    # 展平 topk_weights: [num_tokens * top_k]
    flat_weights = topk_weights.view(-1).contiguous()

    output = torch.zeros(num_tokens * top_k, out_dim,
                         device=hidden_states.device, dtype=torch.float16)

    EM = sorted_ids.shape[0]

    grid = lambda META: (
        triton.cdiv(EM, META['BLOCK_M']) * triton.cdiv(out_dim, META['BLOCK_N']),
    )

    moe_kernel_v4[grid](
        hidden_states, expert_weights, output,
        flat_weights,
        sorted_ids, expert_ids_block, num_padded,
        out_dim, hidden_dim, EM,
        num_tokens * top_k,
        hidden_states.stride(0), hidden_states.stride(1),
        expert_weights.stride(0), expert_weights.stride(1), expert_weights.stride(2),
        output.stride(0), output.stride(1),
        top_k=top_k,
        MUL_ROUTED_WEIGHT=True,
    )

    # 已经乘了 weight, 直接 sum
    output = output.view(num_tokens, top_k, out_dim)
    result = output.sum(dim=1)
    return result

# 验证
v4_output = moe_triton_v4(hidden_states[:8], expert_weights, topk_ids[:8], topk_weights[:8])
diff = (v4_output - ref_output).abs().max()
print(f"V4 vs Naive max diff: {diff:.6f}")
assert diff < 0.1, f"差异过大: {diff}"
print("✓ V4 验证通过")

# Benchmark
torch.cuda.synchronize()
for _ in range(10):
    moe_triton_v4(hidden_states, expert_weights, topk_ids, topk_weights)
torch.cuda.synchronize()
t0 = time.time()
for _ in range(100):
    moe_triton_v4(hidden_states, expert_weights, topk_ids, topk_weights)
torch.cuda.synchronize()
v4_time = (time.time() - t0) / 100
print(f"V4 Triton {num_tokens} tokens: {v4_time*1000:.3f} ms")

## 7.7 对比 vLLM 的实际 fused_moe_kernel

让我们把 vLLM 的完整 kernel 与我们的实现做对比：

```
vLLM fused_moe_kernel 的完整特性:
┌─────────────────────────────────────────────────────────────────┐
│                                                                 │
│  我们实现了:                     vLLM 额外有:                   │
│  ├── sorted_token_ids 路由  ✓    ├── FP8 w8a8 量化              │
│  ├── expert_ids 选择        ✓    ├── INT8 w8a8/w8a16 量化       │
│  ├── Swizzle 调度           ✓    ├── Block-wise 量化 scale      │
│  ├── FP32 累加              ✓    ├── Channel-wise 量化 scale    │
│  ├── MUL_ROUTED_WEIGHT      ✓    ├── Tensor-wise 量化 scale     │
│  ├── Autotune               ✓    ├── HAS_BIAS 支持              │
│  └── 边界检查               ✓    ├── SPLIT_K (未来?)            │
│                                   ├── naive_block_assignment     │
│                                   └── Expert 并行 (-1 跳过)     │
│                                                                 │
│  核心 GEMM 循环: 完全相同!                                      │
│    for k: acc += tl.dot(a, b)                                   │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### 关键差异: 量化路径

vLLM 的 kernel 支持 3 种量化粒度，每种对 GEMM 循环有不同影响：

In [ ]:
# vLLM fused_moe_kernel 中量化的 3 种路径 (伪代码)

print("=== 路径 1: Block-wise 量化 (group_k > 0 and group_n > 0) ===")
print("""
for k in range(K // BLOCK_K):
    a = tl.load(a_ptrs)  # FP8
    b = tl.load(b_ptrs)  # FP8
    # 每个 K-block 有独立的 scale
    a_scale = tl.load(a_scale_ptrs + k_offset)
    b_scale = tl.load(b_scale_ptrs + k_offset)
    acc += tl.dot(a, b) * a_scale[:, None] * b_scale[None, :]
    # ↑ 注意: 不能用 acc=accumulator, 因为 scale 随 k 变化
""")

print("=== 路径 2: Channel/Token-wise 量化 ===")
print("""
# scale 不随 k 变化, 在循环外加载
a_scale = tl.load(a_scale_ptrs)  # per-token
b_scale = tl.load(b_scale_ptrs)  # per-channel
for k in range(K // BLOCK_K):
    a = tl.load(a_ptrs)  # FP8
    b = tl.load(b_ptrs)  # FP8
    acc = tl.dot(a, b, acc=acc)   # FP8 fast accumulate
# 循环结束后统一 dequant:
acc = acc * a_scale[:, None] * b_scale[None, :]
""")

print("=== 路径 3: Tensor-wise 量化 ===")
print("""
a_scale = tl.load(a_scale_ptr)  # 标量
b_scale = tl.load(b_scale_ptr + expert_id)  # 标量
for k in range(K // BLOCK_K):
    acc = tl.dot(a, b, acc=acc)
acc = acc * a_scale * b_scale
""")

print()
print("核心观察: GEMM 循环本身不变, 只有 scale 的加载和应用位置不同")

## 7.8 性能 Benchmark 总结

In [ ]:
print("=" * 70)
print(f"{'方法':>25} | {'时间 (ms)':>12} | {'说明'}")
print("-" * 70)
print(f"{'Naive (8 tokens)':>25} | {naive_time*1000:>12.2f} | 循环调用 cuBLAS")
print(f"{'Grouped cuBLAS':>25} | {grouped_time*1000:>12.3f} | 按 expert 分组")
print(f"{'V3 Triton Basic':>25} | {v3_time*1000:>12.3f} | 单 kernel, 基础版")
print(f"{'V4 Triton + Weight':>25} | {v4_time*1000:>12.3f} | + 融合 router weight")
print("=" * 70)
print()
print("关键结论:")
print("  1. 单 kernel 消除了多次 launch 开销")
print("  2. 融合 router weight 省掉了额外的 element-wise kernel")
print("  3. Autotune 自动选择最优 block size")
print("  4. 真正的性能提升来自: 避免 scatter/gather 的内存搬运")

## 7.9 与 03_matmul_optimization 终极 GEMM 的对比

`03_matmul_optimization/18_matmul_ultimate.ipynb` 中的终极 Triton GEMM
和 vLLM 的 `fused_moe_kernel` 对比：

| 特性 | 终极 GEMM (Ch.18) | fused_moe_kernel |
|------|-------------------|------------------|
| 核心循环 | `acc += tl.dot(a, b)` | `acc += tl.dot(a, b)` **相同** |
| 内存访问 | `make_block_ptr` (结构化) | 普通指针 + mask (间接寻址) |
| Swizzle | ✓ GROUP_SIZE_M | ✓ GROUP_SIZE_M **相同** |
| Pipeline | ✓ num_stages | ✓ num_stages (通过 autotune) |
| Autotune | ✓ | ✓ |
| FP32 累加 | ✓ | ✓ |
| **Token 路由** | ✗ | ✓ sorted_token_ids |
| **Expert 选择** | ✗ | ✓ expert_ids |
| **量化支持** | ✗ | ✓ FP8/INT8/INT4 |
| **Router Weight** | ✗ | ✓ MUL_ROUTED_WEIGHT |
| **用途** | 通用 GEMM benchmark | MoE 专用 |

### 为什么 MoE kernel 不用 make_block_ptr？

`make_block_ptr` 要求内存访问是 **规则的块状模式** (连续行/列)。
但 MoE 的 A 矩阵通过 `sorted_token_ids` 做 **间接寻址**，
每行可能来自不同的 token — 这与 `make_block_ptr` 的假设冲突。

> 这是 vLLM MoE kernel 相比标准 GEMM 性能稍低的主要原因之一。
> 间接寻址导致 L2 cache miss 增加，编译器也无法做某些 smem 优化。

## 7.10 源码映射表

| 本章内容 | vLLM 源码位置 | 说明 |
|----------|--------------|------|
| Token 排序 | `vllm/model_executor/layers/fused_moe/moe_align_block_size.py` | `moe_align_block_size()` |
| Fused MoE Kernel | `vllm/model_executor/layers/fused_moe/fused_moe.py:315` | `fused_moe_kernel` |
| Kernel 调用入口 | `vllm/model_executor/layers/fused_moe/fused_moe.py:728` | `invoke_fused_moe_triton_kernel` |
| Batched MoE | `vllm/model_executor/layers/fused_moe/fused_batched_moe.py:39` | `moe_mmk` 子函数 |
| Expert Kernel | `vllm/model_executor/layers/fused_moe/fused_batched_moe.py:157` | `expert_triton_kernel` |
| GPT-OSS MoE | `vllm/model_executor/layers/fused_moe/gpt_oss_triton_kernels_moe.py` | `matmul_ogs` |
| 配置文件 | `vllm/model_executor/layers/fused_moe/fused_moe.py:1020` | `get_config_file_name` |
| 终极 GEMM | `03_matmul_optimization/18_matmul_ultimate.ipynb` | 标准 Triton GEMM |

### 下一章预告

> Notebook 08 将深入 Triton 注意力 GEMM — decode 和 prefill 两条路径，
> 以及 MLA 专用的 BLOCK_DPE 分离机制。